# Two-stage transfer learning

This notebook reads measured artifacts and reusable source modules. Expensive training is explicit and disabled by default.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'configs/config.yaml').exists(), 'Open this notebook inside the project'
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
from src.utils import load_config
config = load_config()

First freeze the ImageNet backbone and learn the head. Then unfreeze later MobileNetV2 blocks or ResNet50 layer4 at a much lower learning rate. Keep pretrained batch-normalization statistics fixed. Fine-tuning starts from the best frozen checkpoint.

In [ ]:
from src.models import build_model, configure_phase
for name in ['mobilenetv2', 'resnet50']:
    model = build_model(name, pretrained=False)
    for fine_tune in [False, True]:
        configure_phase(model, name, finetune=fine_tune)
        print(name, 'fine_tune=', fine_tune, 'trainable parameters=', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    for name in ['mobilenetv2', 'resnet50']:
        subprocess.run([sys.executable, '-m', 'src.train', '--model', name], cwd=ROOT, check=True)

In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / 'results/model_comparison.csv'))
print(json.loads((ROOT / 'results/selection.json').read_text()))

In [ ]:
for name in ['mobilenetv2_frozen', 'mobilenetv2_finetuned', 'resnet50_frozen', 'resnet50_finetuned']:
    display(Image(filename=str(ROOT / f'results/figures/{name}_history.png'))) 